# 실습예제 2-3 · 형태소 분석: 한국어는 띄어쓰기로 자르면 안 된다

한국어는 조사·어미가 붙는 **교착어**라서 `영화가 · 영화는 · 영화를`이 띄어쓰기 기준으로는 서로 다른 단어로 집계됩니다.
형태소 분석기 **Kiwi**(`kiwipiepy`)로 이 문제를 해결해 봅니다.

### 목차
1. 환경 설정 – Kiwi 설치
2. (방식1) 띄어쓰기 기준 토큰화
3. (방식2) 형태소 기준 토큰화 (Kiwi)
4. 두 방식의 빈도 비교
5. 품사 필터 규칙 만들기
   - ① 명사(NNG·NNP): 2글자 이상만 남김
   - ② 동사·형용사(VV·VA): 어간 + '다'로 기본형 복원
   - ③ 버리는 품사: 조사(J\*), 어미(E\*), 접사(XS\*), 기호(S\*)
6. 영화 리뷰 전체에 적용하고 빈도 상위 단어 보기
7. 정리


## 1. 환경 설정 – Kiwi 설치
Colab에는 Kiwi가 기본 설치되어 있지 않으므로 먼저 설치합니다. (약 10~20초)

In [ ]:
# 한국어 형태소 분석기 Kiwi 설치 (-q: 설치 로그를 간단히)
!pip install -q kiwipiepy

In [ ]:
from collections import Counter   # 단어 빈도를 세기 위한 도구
from kiwipiepy import Kiwi          # 한국어 형태소 분석기

kiwi = Kiwi()                       # 분석기 객체 생성 (처음 한 번만 만들면 됨)
print("Kiwi 준비 완료")

## 2. (방식1) 띄어쓰기 기준 토큰화
가장 단순한 방법: 공백(스페이스)으로 문장을 자릅니다.

In [ ]:
# 실습 입력 문장
text = "영화가 재밌다. 영화는 길었지만 영화를 또 보고 싶다"

# 방식1: 띄어쓰기 기준으로 자르기
space_tokens = text.split()
print(" | ".join(space_tokens))

# '영화'가 들어간 토큰만 모아 보기
print("\n'영화' 관련 토큰:", [t for t in space_tokens if t.startswith("영화")])
# → '영화가', '영화는', '영화를' 3개의 서로 다른 단어로 흩어짐!

## 3. (방식2) 형태소 기준 토큰화 (Kiwi)
`kiwi.tokenize()`는 문장을 **형태소(뜻을 가진 가장 작은 단위)** 와 **품사 태그**로 나눠 줍니다.

In [ ]:
# 방식2: Kiwi 형태소 분석
tokens = kiwi.tokenize(text)

# 각 토큰은 form(형태), tag(품사) 속성을 가짐 → '형태/품사' 형식으로 출력
print("  ".join(f"{t.form}/{t.tag}" for t in tokens))

주요 품사 태그 읽는 법

| 태그 | 의미 | 예 |
|---|---|---|
| NNG / NNP | 일반명사 / 고유명사 | 영화, 배우 |
| VV / VA | 동사 / 형용사 | 보, 재밌 |
| JKS·JKO·JX… (J\*) | 조사 | 가, 를, 는 |
| EF·EC·EP… (E\*) | 어미 | 다, 지만, 었 |
| XSA·XSV… (XS\*) | 접미사 | 하 |
| SF·SP… (S\*) | 기호 | . , ! |


## 4. 두 방식의 빈도 비교

In [ ]:
# 방식1 빈도: 마침표 등 기호가 붙은 채로 집계됨
space_count = Counter(space_tokens)

# 방식2 빈도: 형태소 단위로 집계
morph_count = Counter(t.form for t in tokens)

print("[방식1] 띄어쓰기 기준 :", {k: v for k, v in space_count.items() if "영화" in k})
print("[방식2] 형태소 기준   :", {"영화": morph_count["영화"]})
# → 방식2에서는 '영화' 3회로 정확히 집계됨

## 5. 품사 필터 규칙 만들기

분석 결과를 그대로 쓰면 `가, 는, 다, .` 같은 의미 없는 형태소가 섞입니다. 다음 규칙으로 걸러 냅니다.

1. **명사(NNG·NNP)**: 2글자 이상만 남김 → 배우, 연기, 스토리
2. **동사·형용사(VV·VA)**: 어간 + `'다'`로 복원 → 좋 → 좋다, 지루하 → 지루하다
3. **버리는 품사**: 조사(J\*), 어미(E\*), 접사(XS\*), 기호(S\*)

> 💡 `'좋다'`처럼 어간이 한 글자인 용언은 **길이 필터에 걸리지 않게** 기본형으로 복원해서 남깁니다.


In [ ]:
KEEP_NOUN = {"NNG", "NNP"}           # ① 남길 명사 태그
KEEP_VERB = {"VV", "VA"}             # ② 남길 동사·형용사 태그
DROP_PREFIX = ("J", "E", "XS", "S")  # ③ 버릴 품사 (접두어로 판별)

def tokenize_filtered(sentence):
    """문장을 형태소 분석한 뒤 품사 규칙에 맞는 단어만 리스트로 반환"""
    tokens = kiwi.tokenize(sentence)
    words = []
    for i, t in enumerate(tokens):
        # ① 명사: 2글자 이상만 남김 (한 글자 명사 '것', '수' 등은 제외)
        if t.tag in KEEP_NOUN:
            if len(t.form) >= 2:
                words.append(t.form)

        # ② 동사·형용사: 어간에 '다'를 붙여 기본형으로 복원
        elif t.tag in KEEP_VERB:
            words.append(t.form + "다")          # 좋 → 좋다, 재밌 → 재밌다

        # ②-보충: '지루/XR + 하/XSA'처럼 어근+접사로 쪼개진 형용사 처리
        #          → 어근과 '하'를 합쳐 '지루하다'로 복원
        elif t.tag == "XR":
            nxt = tokens[i + 1] if i + 1 < len(tokens) else None
            if nxt is not None and nxt.tag in ("XSA", "XSV"):
                words.append(t.form + nxt.form + "다")
            elif len(t.form) >= 2:
                words.append(t.form)

        # ③ 조사·어미·접사·기호 및 그 밖의 품사(부사, 관형사 등)는 버림
        elif t.tag.startswith(DROP_PREFIX):
            continue
    return words

# 테스트
print(tokenize_filtered(text))
print(tokenize_filtered("배우의 연기가 좋았고 스토리는 지루하다"))

## 6. 영화 리뷰 전체에 적용하고 빈도 상위 단어 보기

In [ ]:
reviews = [
    "영화가 재밌다. 영화는 길었지만 영화를 또 보고 싶다",
    "배우들의 연기가 정말 좋았고 스토리도 탄탄한 영화였다",
    "이 영화 진짜 꿀잼! 가성비 좋은 꿀잼 영화 추천합니다",
    "스토리가 너무 지루하다. 평점을 높게 준 이유를 모르겠다",
    "재미있다는 평이 많아서 봤는데 생각보다 재미있었다",
    "연출은 좋았지만 결말이 아쉬운 영화. 평점 7점",
    "배우 연기는 최고였는데 스토리는 조금 지루했다",
    "재밌어요 재밌어요! 가족과 함께 보기 좋은 영화",
    "음악과 영상미가 좋아서 극장에서 볼 만한 영화였다",
    "기대를 많이 했는데 노잼이었다. 평점이 아깝다",
    "주연 배우의 연기 덕분에 끝까지 몰입해서 본 영화",
    "이런 것 정도는 누구나 만들 수 있다. 시간이 아깝다",
]

# (방식1) 띄어쓰기 기준 빈도 상위 10
space_all = Counter(w for r in reviews for w in r.split())
print("[방식1] 띄어쓰기 기준 상위 10")
print(space_all.most_common(10))

# (방식2) 형태소 + 품사 필터 기준 빈도 상위 10
morph_all = Counter(w for r in reviews for w in tokenize_filtered(r))
print("\n[방식2] 형태소 + 품사 필터 상위 10")
print(morph_all.most_common(10))

In [ ]:
# 리뷰별 토큰화 결과를 표로 확인 (pandas는 Colab에 기본 설치되어 있음)
import pandas as pd

df = pd.DataFrame({
    "리뷰": reviews,
    "띄어쓰기 토큰": [r.split() for r in reviews],
    "형태소 토큰(필터)": [tokenize_filtered(r) for r in reviews],
})
pd.set_option("display.max_colwidth", None)
df

## 7. 정리
- 한국어는 **띄어쓰기로 자르면** 같은 단어가 조사·어미 때문에 여러 개로 흩어진다.
- **형태소 분석(Kiwi)** 을 쓰면 `영화/NNG + 가/JKS`처럼 분리되어 정확히 집계된다.
- 품사 규칙: **명사 2글자 이상**, **용언은 '다' 붙여 기본형 복원**, **조사·어미·접사·기호는 버림**.
- 다음 실습(2-4)에서는 남은 단어 중 **불용어 제거·표기 통일·신조어 등록**을 다룹니다.
